# QHY42 / SUMMER — remote control examples

Launch the GUI first (camera connected **before** launch — no hotplug):

```bash
python -m cmos_camera_gui --ws-port 8765
# or telescope service mode:
python -m cmos_camera_gui --headless --ws-port 5566 --auto-connect qhy --instrument summer
```

## Native client (lab use)

Exposure in **microseconds**; `camera_state` follows the pirt vocabulary
(`READY / EXPOSING / SAVING / TEC_SETTLING / ERROR / ...`).

In [ ]:
from cmos_camera_gui.client import CameraClient

cam = CameraClient('ws://localhost:8765')
cam.connect()
print(cam.list_cameras())
cam.connect_camera(0)   # QHY42PRO shows vendor='qhy'

In [ ]:
import json
print(json.dumps(cam.status(), indent=2))  # camera_state, tec_locked, gps, idle_mode, ...

### Grab a gated stack with a combined output

Every kept exposure starts *after* the record command (transition and
in-flight frames are discarded by the record gate). GPS seq/UTC per
frame land in a `FRAMEMETA` bintable of the cube file.

In [ ]:
cam.set(Exposure=300_000, Gain=10)   # 300 ms
cam.start_stream()
done = cam.capture_frames(10, directory='./captures', basename='qhy_demo',
                          combine='mean')   # also writes qhy_demo_mean.fits
print(done['message'])

### Summed 'synthetic long exposure'

`combine='sum', combine_only=True` writes one float32 frame whose
`EXPTIME` is the net integration (per-frame exposure moves to `EXPFRAME`).
Calibrate with darks summed the same way — the pedestal sums too.

In [ ]:
done = cam.capture_frames(100, basename='sum10s', combine='sum', combine_only=True)
print(done['message'])   # 100 x 0.1 s -> EXPTIME = 10000 ms

### High-speed idle (telescope ops)

Between grabs the stream idles at 1 ms instead of the target exposure.
Grab latency floor (T + ~65 ms) is achieved either way on the QHY42;
idle mode controls the between-grab readout duty cycle.

In [ ]:
cam.idle_mode(True, idle_exposure_us=1000)
cam.set(Exposure=2_000_000)      # staged; applied at the next grab
cam.capture_frames(5, basename='idle_demo')
cam.idle_mode(False)

### TEC + telescope-style state polling

In [ ]:
cam.cooler(on=True, target=0)
cam.wait_for_state('READY', timeout=300)   # TEC_SETTLING -> READY once stable
print(cam.status()['tec_locked'])

## WSP / SUMMER client (pirtcam-compatible)

Float **seconds**, ack-then-poll captures, exact `<save_path>/<filename>.fits`
output (2D for `nframes=1`, single cube file for `nframes>1`).
See `docs/WspSummerDaemonHandoff.md` for the daemon-side contract.

In [ ]:
import time
from cmos_camera_gui.summer_client import SummerClient

scam = SummerClient(host='localhost', port=8765)
scam.connect()
scam.set_exposure(2.0)
scam.set_save_path('./captures/wsp')
scam.capture_frames(filename='img_0001', nframes=1, object='TEST',
                    observer='notebook', headers=[['FIELDID', 42, 'test card']])
while scam.get_status()['data']['is_capturing']:
    time.sleep(1)
print('done:', scam.get_status()['data']['camera_state'])